In [1]:
import pandas as pd

df_csv = pd.read_csv("starter-kit/data/numeros_tva.csv", keep_default_na=False)
df_xlsx = pd.read_excel("starter-kit/data/numeros_tva.xlsx", keep_default_na=False)

print("Dimensions :", df_csv.shape)
print("Différences CSV vs Excel :")
print((df_csv != df_xlsx).sum())
df_csv.head()

Dimensions : (10000, 6)
Différences CSV vs Excel :
id                0
raison_sociale    0
pays_declare      0
numero_tva        0
date_saisie       0
source_saisie     0
dtype: int64


,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie
0,1,Nord Technologies GmbH,DK,DK73224645,2025-07-27,portail_client
1,2,Solaris Technologies SARL,PT,PT787327239,2024-03-29,reprise_erp
2,3,Verso Consulting NV,LU,LU71043234,2024-02-16,crm
3,4,Lumen Distribution Sp. z o.o.,SE,SE.751298146001,2024-02-27,import_fournisseur
4,5,Textil Services Oy,NL,NL505862176B88,2025-04-14,crm


In [2]:
print("Top 15 des valeurs dans numero_tva :")
print(df_csv["numero_tva"].value_counts().head(15))

print("\nCases vides ou espaces par colonne :")
for col in df_csv.columns:
    vides = (df_csv[col].astype(str).str.strip() == "").sum()
    print(f"- {col} : {vides}")

Top 15 des valeurs dans numero_tva :
numero_tva
                   60
                   59
-                  55
N/A                48
null               38
0282330178          4
PT823046656         3
DK73224645          2
DK48646344          2
BE0605238824        2
FI4715363           2
SE593651753601      2
LU75426276          2
IT 5368 0520532     2
nl785427788b39      2
Name: count, dtype: int64

Cases vides ou espaces par colonne :
- id : 0
- raison_sociale : 0
- pays_declare : 0
- numero_tva : 119
- date_saisie : 0
- source_saisie : 0


In [3]:
# 1. Nombre et liste des pays déclarés dans le CRM
print("Pays déclarés distincts :", df_csv["pays_declare"].nunique())
print(df_csv["pays_declare"].value_counts())

# 2. Comparer pays_declare avec les 2 premières lettres du numéro nettoyé
prefixes = df_csv["numero_tva"].str.strip().str.upper().str[:2]
incoherences = df_csv[prefixes != df_csv["pays_declare"]]
print("\nLignes où préfixe != pays_declare :", len(incoherences))
print(incoherences[["id", "pays_declare", "numero_tva"]].head(15))

# 3. Proportion de lignes contenant des caractères parasites (espaces, tirets, points, minuscules...)
avec_parasites = df_csv["numero_tva"].str.contains(r"[^A-Z0-9]", regex=True)
print(f"\nLignes avec caractères parasites : {avec_parasites.sum()} ({avec_parasites.mean():.1%})")

Pays déclarés distincts : 15
pays_declare
FR    974
DK    961
BE    959
LU    956
SE    949
PT    947
NL    947
IT    947
PL    939
FI    902
ZZ    115
QQ    109
GB    104
UK    104
XX     87
Name: count, dtype: int64

Lignes où préfixe != pays_declare : 793
      id pays_declare    numero_tva
25    26           PT             -
80    81           IT              
91    92           FR              
119  120           FR   58560322811
134  135           NL  396904956B68
162  163           IT   38486825359
180  181           IT   33139885439
188  189           IT   88838354509
194  195           SE  107327005801
246  247           PL    5112134613
259  260           LU      75041556
261  262           BE    0572894866
263  264           IT   09184526433
267  268           IT              
269  270           LU      22312502

Lignes avec caractères parasites : 2341 (23.4%)


In [4]:
# 1. Répartition des 10 000 lignes par canal d'origine
print("Répartition par canal (source_saisie) :")
print(df_csv["source_saisie"].value_counts())

# 2. Plage de dates et vérification du format
print("\nPlage de dates :", df_csv["date_saisie"].min(), "->", df_csv["date_saisie"].max())
dates_valides = pd.to_datetime(df_csv["date_saisie"], errors="coerce")
print("Dates invalides :", dates_valides.isna().sum())

# 3. Taux de numéros vides par canal
est_vide = df_csv["numero_tva"].str.strip().str.upper().isin(["", "-", "N/A", "NULL"])
print("\nTaux de numéros vides par canal :")
print((est_vide.groupby(df_csv["source_saisie"]).mean() * 100).round(2))

Répartition par canal (source_saisie) :
source_saisie
reprise_erp           2071
crm                   2019
portail_client        1990
saisie_manuelle       1976
import_fournisseur    1944
Name: count, dtype: int64

Plage de dates : 2024-01-01 -> 2025-10-01
Dates invalides : 0

Taux de numéros vides par canal :
source_saisie
crm                   2.82
import_fournisseur    2.88
portail_client        2.36
reprise_erp           2.85
saisie_manuelle       2.07
Name: numero_tva, dtype: float64


260 vides sous 5 formes, 15 pays dont 5 hors périmètre, 23,4 % de bruit de saisie, préfixes manquants